In [2]:
def generate_contact_map(fasta_path, sequence):
    """
    Executes an external ESM-2 script to generate a contact map from a FASTA file.
    The resulting CSV is stored in results_esm2/contact_maps/. If the file already
    exists, the computation is skipped.
    """

    contact_maps_dir = os.path.join("predicted_contact_maps")
    os.makedirs(contact_maps_dir, exist_ok=True)
    
    csv_path = os.path.join(contact_maps_dir, f"{sequence}.csv")
    
    if os.path.exists(csv_path):
        print(f"Skipping: contact map already exists at {csv_path}")
        return csv_path
    
    python_env = "/home/biocomp/anaconda3/envs/esmfold/bin/python"
    script_path = os.path.join("esm2", "get_contact_map.py")
    
    print(f"Running ESM-2 contact map generation for sequence: {sequence}")
    
    try:
        subprocess.run([python_env, script_path, fasta_path, csv_path], check=True)
        print(f"Contact map successfully generated at: {csv_path}")
        return csv_path
        
    except subprocess.CalledProcessError:
        print("Error: ESM-2 script execution failed.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    
    return None

In [3]:
import os
import glob
import subprocess
from pathlib import Path

def process_all_fastas(input_folder):
    fasta_files = glob.glob(os.path.join(input_folder, "*.fasta"))
    
    if not fasta_files:
        print(f"No se encontraron archivos .fasta en {input_folder}")
        return

    print(f"Se encontraron {len(fasta_files)} archivos. Iniciando procesamiento...")

    for fasta_path in fasta_files:

        sequence_name = Path(fasta_path).stem
        
        generate_contact_map(fasta_path, sequence_name)


In [5]:
directorio_fastas = "fasta_files"
process_all_fastas(directorio_fastas)

Se encontraron 31 archivos. Iniciando procesamiento...
Running ESM-2 contact map generation for sequence: 5uiy
Loading ESM-2 model (this may take a few seconds)...
Predicting contacts for sequence of length 19...
Success! ESM-2 matrix (19x19) saved with headers to: predicted_contact_maps/5uiy.csv
Contact map successfully generated at: predicted_contact_maps/5uiy.csv
Running ESM-2 contact map generation for sequence: 1emn
Loading ESM-2 model (this may take a few seconds)...
Predicting contacts for sequence of length 27...
Success! ESM-2 matrix (27x27) saved with headers to: predicted_contact_maps/1emn.csv
Contact map successfully generated at: predicted_contact_maps/1emn.csv
Running ESM-2 contact map generation for sequence: 1llm
Loading ESM-2 model (this may take a few seconds)...
Predicting contacts for sequence of length 29...
Success! ESM-2 matrix (29x29) saved with headers to: predicted_contact_maps/1llm.csv
Contact map successfully generated at: predicted_contact_maps/1llm.csv
Run

In [7]:
import os
import pandas as pd
import glob

def summarize_contact_maps(directory="predicted_contact_maps"):
    # Lista para almacenar los datos de cada archivo
    summary_data = []
    
    # Buscamos todos los archivos CSV generados
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    
    if not csv_files:
        print(f"No se encontraron archivos CSV en {directory}")
        return
    
    print(f"Procesando {len(csv_files)} archivos...")

    for file_path in csv_files:
        try:
            # Leer el CSV (asumiendo que no tiene encabezados si es una matriz pura)
            # Si tu script de ESM2 guarda con headers, usa: pd.read_csv(file_path)
            df = pd.read_csv(file_path)
            
            # 1. Nombre de la secuencia (nombre del archivo sin extensión)
            sequence_name = os.path.basename(file_path).replace(".csv", "")
            
            # 2. Longitud (número de filas)
            length = len(df)
            
            # 3. Probabilidades mayores a cero
            # Convertimos a numérico por si acaso y contamos valores > 0
            count_positive = (df > 0).sum().sum()
            
            summary_data.append({
                "Nombre de la secuencia": sequence_name,
                "Longitud": length,
                "# Probabilidades > 0": count_positive
            })
            
        except Exception as e:
            print(f"Error procesando {file_path}: {e}")

    # Crear el DataFrame final
    summary_df = pd.DataFrame(summary_data)
    
    # Opcional: Ordenar por nombre
    summary_df = summary_df.sort_values("Nombre de la secuencia")
    
    # Guardar a un archivo CSV final
    output_name = "resumen_contactos.csv"
    summary_df.to_csv(output_name, index=False)
    
    print(f"\nTabla generada exitosamente: {output_name}")
    print(summary_df.head()) # Mostrar las primeras filas en consola

summarize_contact_maps()

Procesando 31 archivos...

Tabla generada exitosamente: resumen_contactos.csv
   Nombre de la secuencia  Longitud  # Probabilidades > 0
13                   1aay        26                   629
7                    1emn        27                   685
17                   1f2i        28                   727
14                   1llm        29                   792
5                    1lql        26                   624


In [10]:
import os
import glob
import pandas as pd
import numpy as np

def summarize_contact_maps_advanced(directory="predicted_contact_maps"):
    summary_data = []
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    
    if not csv_files:
        print(f"No se encontraron archivos CSV en {directory}")
        return
    
    print(f"Procesando {len(csv_files)} archivos...")

    for file_path in csv_files:
        try:
            # 1. Leer el CSV y convertirlo a matriz de NumPy para cálculos rápidos
            df = pd.read_csv(file_path)
            mat = df.to_numpy()
            
            sequence_name = os.path.basename(file_path).replace(".csv", "")
            length = len(df)
            
            # 2. Crear una matriz de distancias |i - j|
            # Esto genera una matriz NxN donde cada celda indica la distancia en la secuencia
            indices = np.arange(length)
            dist_matrix = np.abs(indices[:, None] - indices)
            print(dist_matrix)
            # 3. Crear las máscaras lógicas para las probabilidades > 0
            is_positive = (mat > 0)
            
            # Totales
            count_total = is_positive.sum()
            
            # Sin diagonal: |i - j| > 0
            count_no_diag = (is_positive & (dist_matrix > 0)).sum()
            
            # Sin distancia 1 (se excluye c_i y c_i+1): |i - j| > 1
            count_dist_gt_1 = (is_positive & (dist_matrix > 1)).sum()
            
            # Sin distancia 2 (se excluye hasta c_i+2): |i - j| > 2
            count_dist_gt_2 = (is_positive & (dist_matrix > 2)).sum()
            
            # Sin distancia 3 (se excluye hasta c_i+3): |i - j| > 3
            count_dist_gt_3 = (is_positive & (dist_matrix > 3)).sum()
            
            # Agregar a los resultados
            summary_data.append({
                "Nombre de la secuencia": sequence_name,
                "Longitud": length,
                "# Prob > 0 (Total)": count_total,
                "# Prob > 0 (Sin diagonal)": count_no_diag,
                "# Prob > 0 (Distancia > 1)": count_dist_gt_1,
                "# Prob > 0 (Distancia > 2)": count_dist_gt_2,
                "# Prob > 0 (Distancia > 3)": count_dist_gt_3
            })
            
        except Exception as e:
            print(f"Error procesando {file_path}: {e}")

    # Crear el DataFrame y exportar
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values("Nombre de la secuencia")
    
    output_name = "resumen_contactos_detallado.csv"
    summary_df.to_csv(output_name, index=False)
    
    print(f"\nTabla detallada generada exitosamente: {output_name}")
    print(summary_df.head())

summarize_contact_maps_advanced()

Procesando 31 archivos...
[[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
 [ 1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
 [ 2  1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
 [ 3  2  1  0  1  2  3  4  5  6  7  8  9 10 11 12 13]
 [ 4  3  2  1  0  1  2  3  4  5  6  7  8  9 10 11 12]
 [ 5  4  3  2  1  0  1  2  3  4  5  6  7  8  9 10 11]
 [ 6  5  4  3  2  1  0  1  2  3  4  5  6  7  8  9 10]
 [ 7  6  5  4  3  2  1  0  1  2  3  4  5  6  7  8  9]
 [ 8  7  6  5  4  3  2  1  0  1  2  3  4  5  6  7  8]
 [ 9  8  7  6  5  4  3  2  1  0  1  2  3  4  5  6  7]
 [10  9  8  7  6  5  4  3  2  1  0  1  2  3  4  5  6]
 [11 10  9  8  7  6  5  4  3  2  1  0  1  2  3  4  5]
 [12 11 10  9  8  7  6  5  4  3  2  1  0  1  2  3  4]
 [13 12 11 10  9  8  7  6  5  4  3  2  1  0  1  2  3]
 [14 13 12 11 10  9  8  7  6  5  4  3  2  1  0  1  2]
 [15 14 13 12 11 10  9  8  7  6  5  4  3  2  1  0  1]
 [16 15 14 13 12 11 10  9  8  7  6  5  4  3  2  1  0]]
[[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17

In [14]:
import os
import numpy as np
import pandas as pd

def fitness_shannon(predicted_csv, real_csv, pruebas_dir="pruebas", grid_size=5):
    
    # 1. Definir rutas y cargar archivos
    predicted_path = os.path.join(pruebas_dir, predicted_csv)
    real_path = os.path.join(pruebas_dir, real_csv)
    
    if not os.path.exists(predicted_path) or not os.path.exists(real_path):
        print("Error: No se encontraron los archivos CSV en la carpeta especificada.")
        return 0.0
        
    probability_matrix = pd.read_csv(predicted_path).to_numpy()
    real_matrix = pd.read_csv(real_path).to_numpy()
    
    L = probability_matrix.shape[0] # Longitud de la cadena

    if L == 0 or probability_matrix.shape != real_matrix.shape:
        print(f"Error: Discrepancia de dimensiones. Pred:{probability_matrix.shape}, Real:{real_matrix.shape}")
        return 0.0

    # 2. Enmascarar contactos obvios (|i - j| <= 2) y el triángulo inferior
    min_separation = 3 # Ignora vecinos hasta i+2
    i_indices, j_indices = np.indices((L, L))
    separation = np.abs(i_indices - j_indices)
    
    obvious_mask = separation < min_separation
    lower_triangle_mask = np.tril(np.ones((L, L)), k=0).astype(bool)  # Se quita el trinagulo inferior para que no se repitan probabilidades
    
    filtered_matrix = probability_matrix.copy()
    filtered_matrix[obvious_mask] = 0.0
    filtered_matrix[lower_triangle_mask] = 0.0

    # 3. Obtener los índices del Top L
    flat_predicted = filtered_matrix.flatten()
    # np.argsort ordena de menor a mayor, tomamos los últimos L y los invertimos
    top_l_flat_indices = np.argsort(flat_predicted)[-L:][::-1]
    
    # Convertir índices 1D de vuelta a coordenadas 2D (filas, columnas)
    rows, cols = np.unravel_index(top_l_flat_indices, (L, L))
    
    # 4. Calcular Precisión Top-L
    # Verificamos cuáles de esas predicciones son 1 en la matriz real
    is_true_positive = real_matrix[rows, cols] == 1
    true_positives_count = np.sum(is_true_positive)
    precision = true_positives_count / L
    
    # 5. Calcular Entropía de Shannon en 25 celdas fijas (5x5)
    total_cells = grid_size * grid_size # Siempre será 25
    
    # Creamos un grid vacío de 5x5 para contar los verdaderos positivos
    T_grid = np.zeros((grid_size, grid_size))
    
    tp_rows = rows[is_true_positive]
    tp_cols = cols[is_true_positive]
    
    for r, c in zip(tp_rows, tp_cols):
        # Mapeamos la coordenada de la matriz original (0 a L-1) a la cuadrícula (0 a 4)
        cell_r = (r * grid_size) // L
        cell_c = (c * grid_size) // L
        T_grid[cell_r, cell_c] += 1
        
    # Calculamos la sumatoria de Shannon: H = - sum(p_e * ln(p_e))
    H = 0.0
    for count in T_grid.flatten():
        if count > 0:
            p_e = count / L
            H -= p_e * np.log(p_e) 

    return  H

In [16]:
fitness_shannon("1aay_predicted_contact.csv", "1aay_contact_map.csv", pruebas_dir="pruebas", grid_size=5)

np.float64(1.5726241695174308)